# 推論回路の同定

| ステップ | 内容 |
|---|---|
| **Phase 0** | 環境とモデル |
| **Phase 1** | 質問の記入 → 検証 → スクリーニング |
| **Phase 2** | 反応度マップ 8枚 |
| **Phase 3** | ヘッド別の直接寄与 と 族をまたいだ交差 |
| **Phase 4** | アブレーションと反復的な絞り込み |

`[公式流用]` と付いたブロックは公式デモの本文をそのままコピーしたもの。

---
## Phase 0: 環境

**VS Code から Colab ランタイムに繋いでいる場合の注意**
公式デモは Colab を検出すると plotly のレンダラを `"colab"` に設定するが、
これは Colab のウェブ画面専用で **VS Code では図が一切描画されない**（エラーも出ない）。
そのため下のセルで `notebook_connected` に上書きしている。

In [1]:
# ========== STEP 1 / 12 : 環境セットアップ + モデル読み込み ==========
# =========================================================
#  メモリを完全に空にしてから始める（import より前に実行すること）
# =========================================================
#  VS Code の「カーネル再起動」は Colab 側の古いプロセスを殺さないため、
#  再起動を繰り返すと GPU を掴んだままのゾンビが積み上がって OOM になる。
#  ここで (1) 他プロセスを終了 (2) 前回の変数を削除 (3) キャッシュ解放 を行う。
#  ※ モジュールは消さない（消すと einops などが未定義になる）

import gc
import os
import signal
import subprocess
import time
import types

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


def _gpu_used_gb():
    try:
        import torch as _t
        if not _t.cuda.is_available():
            return 0.0
        free, total = _t.cuda.mem_get_info()
        return (total - free) / 1e9
    except Exception:
        return 0.0



def _is_heavy(obj):
    """メモリを占有する実体かどうか。クラス・関数・モジュールは対象外。"""
    try:
        import torch as _t
        if isinstance(obj, _t.Tensor):
            return True
        if isinstance(obj, _t.nn.Module):
            return True
    except Exception:
        pass
    if type(obj).__name__ in ("ActivationCache", "TransformerBridge", "HookedTransformer"):
        return True
    if isinstance(obj, (list, tuple)) and obj:
        try:
            import torch as _t
            return isinstance(obj[0], _t.Tensor)
        except Exception:
            return False
    if isinstance(obj, dict) and obj:
        try:
            import torch as _t
            return isinstance(next(iter(obj.values())), _t.Tensor)
        except Exception:
            return False
    return False


def wipe_memory(verbose=True):
    """GPU を空にする。他プロセスの終了 + 前回の変数削除 + キャッシュ解放。"""
    if verbose:
        print(f"[開始] GPU 使用中: {_gpu_used_gb():.2f} GB")

    # (1) 自分以外で GPU を使っているプロセスを終了させる
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30).stdout.strip()
        me = os.getpid()
        killed = 0
        for line in [x for x in out.splitlines() if x.strip()]:
            parts = line.split(",")
            pid = int(parts[0].strip())
            if pid != me:
                if verbose:
                    print(f"  古いプロセス {pid} を終了 ({parts[1].strip()})")
                os.kill(pid, signal.SIGKILL)
                killed += 1
        if killed:
            time.sleep(3)
    except Exception as e:
        if verbose:
            print("  プロセス確認をスキップ:", e)

    # (2) 前回の重い変数を消す（モジュールと関数は残す）
    g = globals()
    for name in [n for n in list(g) if not n.startswith("_")]:
        if _is_heavy(g.get(name)):
            try:
                del g[name]
            except Exception:
                pass

    # (3) IPython が保持している過去のセル出力を消す
    #     Out[] や _ , __ , ___ に古いモデルへの参照が残ると解放されない
    try:
        ip = get_ipython()
        if ip is not None:
            out = ip.user_ns.get("Out")
            if isinstance(out, dict):
                out.clear()
            for nm in ("_", "__", "___", "_i", "_ii", "_iii"):
                if nm in ip.user_ns:
                    ip.user_ns[nm] = None
            ip.displayhook.flush()
    except Exception:
        pass

    # (4) 解放
    for _ in range(3):
        gc.collect()
        try:
            import torch as _t
            if _t.cuda.is_available():
                _t.cuda.empty_cache()
                _t.cuda.ipc_collect()
        except Exception:
            pass

    if verbose:
        print(f"[完了] GPU 使用中: {_gpu_used_gb():.2f} GB")


def free_except(*keep_names, verbose=True):
    """モデルなど指定したものだけ残し、他の重い変数を全部消す。
    STEP をまたいでキャッシュが残り OOM になるのを防ぐ。"""
    keep = set(keep_names)
    g = globals()
    dropped = []
    for name in [n for n in list(g) if not n.startswith("_") and n not in keep]:
        if _is_heavy(g.get(name)):
            try:
                del g[name]
                dropped.append(name)
            except Exception:
                pass
    for _ in range(3):
        gc.collect()
        try:
            import torch as _t
            if _t.cuda.is_available():
                _t.cuda.empty_cache()
                _t.cuda.ipc_collect()
        except Exception:
            pass
    if verbose:
        print(f"解放: {len(dropped)} 個 / GPU 使用中: {_gpu_used_gb():.2f} GB")


wipe_memory()

# ---- [公式流用] Activation_Patching_in_TL_Demo cell 4 ----
# Janky code to do different setup when run in a Colab notebook vs VSCode
DEBUG_MODE = False
try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    %pip install transformer_lens
    # Install my janky personal plotting utils
    %pip install git+https://github.com/neelnanda-io/neel-plotly.git
except:
    IN_COLAB = False
    print("Running as a Jupyter notebook - intended for development only!")
    from IPython import get_ipython

    ipython = get_ipython()
    # Code to automatically update the TransformerBridge code as its edited without restarting the kernel
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

# ---- [公式流用] Activation_Patching_in_TL_Demo cell 5 ----
# Plotly needs a different renderer for VSCode/Notebooks vs Colab argh
import plotly.io as pio

if IN_COLAB or not DEBUG_MODE:
    # Thanks to annoying rendering issues, Plotly graphics will either show up in colab OR Vscode depending on the renderer - this is bad for developing demos! Thus creating a debug mode.
    pio.renderers.default = "colab"
else:
    pio.renderers.default = "png"

# ---- [VS Code 用の上書き] ----
# 上の公式セルは Colab 検出時に "colab" レンダラを選ぶが、VS Code では描画されない。
pio.renderers.default = "notebook_connected"
print("renderer ->", pio.renderers.default)

# ---- [公式流用] Activation_Patching_in_TL_Demo cell 6 ----
# Import stuff
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import einops
from fancy_einsum import einsum
import tqdm.notebook as tqdm
import random
from pathlib import Path
import plotly.express as px
from torch.utils.data import DataLoader

from typing import List, Union, Optional
from functools import partial
import copy

import itertools
from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
import dataclasses
import datasets
from IPython.display import HTML

# ---- [公式流用] Activation_Patching_in_TL_Demo cell 7 ----
# NBVAL_IGNORE_OUTPUT
import transformer_lens
import transformer_lens.utilities as utils
from transformer_lens.model_bridge import TransformerBridge

# ---- [公式流用] Exploratory_Analysis_Demo cell 8 より抜粋 ----
from jaxtyping import Float
from transformer_lens import ActivationCache

# ---- [公式流用] Activation_Patching_in_TL_Demo cell 9 ----
# NBVAL_IGNORE_OUTPUT
_ = torch.set_grad_enabled(False)

# ---- [公式流用] Activation_Patching_in_TL_Demo cell 11 ----
try:
    from neel_plotly import line, imshow, scatter
except ImportError:
    # neel_plotly is an optional visualization dependency.
    # Define no-op stubs so patching computations still run without it.
    def line(*args, **kwargs): pass
    def imshow(*args, **kwargs): pass
    def scatter(*args, **kwargs): pass

# ---- [公式流用] Activation_Patching_in_TL_Demo cell 12 ----
import transformer_lens.patching as patching

# ---- 作図ライブラリの確認 ----
# neel_plotly が無いと上の公式セルは no-op スタブに落ち、エラーも出さず図が出ない。
import importlib.util
if importlib.util.find_spec("neel_plotly") is None:
    raise RuntimeError(
        "neel_plotly が入っていません(図が出ません)。次を実行してカーネルを再起動してください:\n"
        "  %pip install git+https://github.com/neelnanda-io/neel-plotly.git"
    )
print("作図OK: neel_plotly")

# ---- [公式流用] Qwen.ipynb cell 8 の読み込み部分 ----
# 断片化対策(モデル読み込みより前に設定する必要がある)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- モデル ----
# TransformerBridge は HF モデルを保持したまま TL 用の重みも作るため、
# パラメータ数の2〜4倍のVRAMを食う。1.5B は T4(16GB)に載らなかった。
# 素のモデルは書式を理解できなかったので、同じサイズの指示調整版を試す。
# dtype は float32。fp16 だと互換モードの一部の重みが fp32 のまま残り、型が衝突する。
model_path = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerBridge.boot_transformers(model_path, device=device, dtype=torch.float32)
model.enable_compatibility_mode()

if torch.cuda.is_available():
    print(f"読み込み後の使用VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

head_labels = [f"L{l}H{h}" for l in range(model.cfg.n_layers) for h in range(model.cfg.n_heads)]
ALL_HEAD_LABELS = head_labels

print(model_path, "| layers:", model.cfg.n_layers, "heads:", model.cfg.n_heads, "d_model:", model.cfg.d_model)

[開始] GPU 使用中: 0.11 GB
[完了] GPU 使用中: 0.11 GB
Running as a Colab notebook
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.2 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=4f2a126250221e3c5c9c966d7b7374031dbdcd7a7d678fdb4173fb61930acac0
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator
  Cloning https://github.com/neelnanda-io/neel-plotly.git to /tmp/pip-req-build-sajjv1p4
  Running command git clone --filter=blob:none --quiet https://github.com/neelnanda-io/neel-plotly.git /tmp/pip-req-build-sajjv1p4
  Resolved https://github.com/neelnanda-io/neel-plotly.git to commit 6dc24b26f8dec991908479d7445dae496b3430b7
  Preparing metadata (setup.py) ... done
  Created

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning:


Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.



config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

読み込み後の使用VRAM: 2.5 GB
Qwen/Qwen2.5-0.5B-Instruct | layers: 24 heads: 14 d_model: 896


**動作確認**: 下のセルで図が出れば描画設定は正常。出なければ
`pio.renderers.default = "png"` に変えて再実行する（静止画になるが確実に出る）。

In [2]:
# ========== STEP 2 / 12 : 描画テスト（図が出るか確認） ==========
# 描画テスト(これが出なければ以降の図も出ない)
imshow(torch.arange(12).reshape(3, 4).float(), xaxis="X", yaxis="Y", title="描画テスト")

---
## Phase 1: 質問の記入

**次のセルが唯一の入力。** ここ以外に質問は出てこない。

| キー | 意味 |
|---|---|
| `clean` | 正常なプロンプト。`correct` が答えになる |
| `corrupted` | **1箇所だけ**変えた版。答えが `wrong` に反転する |
| `correct` | clean での正解（先頭に半角スペース） |
| `wrong` | corrupted での正解 = clean での不正解 |

**次のセルが自動で検査する条件**
1. `clean` と `corrupted` のトークン長が完全一致（パッチングの前提）
2. `correct` と `wrong` の先頭トークンが異なる
3. 変更箇所が1箇所だけ

違反した事例は理由付きで除外される。族名は自由。

In [2]:
# ========== STEP 3 / 12 : 問題の生成  ★ここに質問を書く★ ==========
# =========================================================
#  設計の根拠（なぜこの質問なのか）
# =========================================================
#
#  目的: Transformer の中で知識と混ざっている推論部分を特定する。
#
#  [1] なぜ領域を1つに絞ったか
#      推論の共通項は「関係を処理すること」(Halford の関係複雑性理論 /
#      Spearman の g 因子の定義)。領域が違っても中身は関係処理なので、
#      動かすべき変数は【関係の数】であって領域ではない。
#      領域を増やすのは「出た回路が表面に依存しない」ことの確認用で、後工程。
#
#  [2] なぜ推移的推論か
#      関係処理の容量を測る正典的課題は「推移的推論」と「ラテン方格」の2つだけ。
#      ラテン方格は図形課題なのでテキストでは使えない。よって推移的推論が残る。
#      3項の推移推論は ternary(関係3個)であり、二項関係の連鎖に還元できない。
#
#  [3] なぜ質問を固定して前提を壊すか
#      パッチングが追うのは「clean と corrupted の違いを運ぶ部品」。
#      質問語を反転させる(tallest <-> shortest)と、順序を組み立てる作業は
#      両方の実行で同一になり差分から消える。追えるのは読み出しの回路だけ。
#      前提を壊せば順序そのものが変わるので、差分に【統合の作業】が乗る。
#
#  [4] なぜ前提の順番をランダムにするか
#      降順に並べると「最大=最初の名前 / 最小=最後の名前」と位置で決まり、
#      統合せずに位置だけで正解できてしまう。順番を崩すとそれが通らなくなる。
#
# ===========================================================
#  N項推移推論 (関係複雑性 0 → 4)
# ===========================================================
#  根拠: Halford らの関係複雑性理論。推移的推論は処理容量を測る正典的課題で、
#        3項の推移推論は ternary(関係3個)であり二項関係の連鎖に還元できない。
#        N項課題はその項数を増やした拡張版。
#
#  領域は推移性ひとつに固定し、関係の数だけを動かす。
#  corrupted は最後の問い方の語を1語だけ入れ替えて答えを反転させる。
#
#    rel0 : 統合なし。並記された事実を引くだけ(対照群)
#    rel1 : A > B                     2項
#    rel2 : A > B > C                 3項   ← 統合が必要
#    rel3 : A > B > C > D             4項   ← 大人の処理限界
#    rel4 : A > B > C > D > E         5項   ← 限界超え
#
#  重要: 名前や比較語のトークン数がバラつくと、バッチ内で長さが揃わず
#        パディングが入り、最終位置が詰め物になって測定が壊れる。
#        そこで語彙をトークナイザで選別してから使う。

import random
random.seed(0)

N_PER_COND = 50

def n_tok(s):
    return model.to_tokens(s, prepend_bos=False).shape[1]

# ---- 名前の候補をトークナイザで選別(単独でも先頭スペース付きでも1トークン) ----
NAME_POOL = ["Ann","Bob","Cid","Dan","Eve","Fred","Gina","Hank","Iris","Jack",
             "Kate","Leo","Mia","Ned","Olga","Paul","Rita","Sam","Tina","Uma",
             "Vera","Walt","Yara","Zane","Cara","Dave","Emma","Finn","Gwen","Hugo",
             "Ivan","Jane","Karl","Lena","Mark","Nina","Oscar","Pete","Quinn","Rose",
             "Tom","Amy","Ben","Joe","Lucy","Nick","Ruth","Adam","Clara","Derek",
             "Ella","Frank","Grace","Henry","Julia","Kevin","Laura","Peter","Sarah","Tony"]
NAMES = [x for x in NAME_POOL if n_tok(x) == 1 and n_tok(" " + x) == 1]

# ---- 比較語の候補を選別(最上位語と最下位語のトークン数が一致するものだけ) ----
# (比較級, 最上級hi, 最上級lo, 原級hi, 原級lo)
# 原級は rel0(単項述語 = 統合不要)で使う
DIM_POOL = [
    ("taller",  "tallest",  "shortest", "tall",   "short"),
    ("older",   "oldest",   "youngest", "old",    "young"),
    ("faster",  "fastest",  "slowest",  "fast",   "slow"),
    ("richer",  "richest",  "poorest",  "rich",   "poor"),
    ("stronger","strongest","weakest",  "strong", "weak"),
    ("bigger",  "biggest",  "smallest", "big",    "small"),
    ("heavier", "heaviest", "lightest", "heavy",  "light"),
    ("happier", "happiest", "saddest",  "happy",  "sad"),
]
DIMS = [d for d in DIM_POOL
        if n_tok(" " + d[1]) == n_tok(" " + d[2])       # 最上級どうし
        and n_tok(" " + d[3]) == n_tok(" " + d[4])]     # 原級どうし

print(f"使える名前 {len(NAMES)} / {len(NAME_POOL)}")
print(f"使える比較語 {len(DIMS)} / {len(DIM_POOL)}  ->", [d[0] for d in DIMS])
assert len(NAMES) >= 6 and len(DIMS) >= 1, "語彙が足りない"
print()



# ---- 手本(few-shot)の前置き ----
# 素のモデルは書式が分からず「直近の名前」を答えるだけになりがち。手本で書式だけを教える。
#
# 重要: 手本は全条件で完全に同一の文字列にする。
#       条件ごとに手本が違うと、成績や図の差が「関係の数のせい」か
#       「手本が違うせい」か区別できなくなり、引き算が成立しない。
#
# 釣り合い: 4問のうち2問は最上位(=最初の名前)、2問は最下位(=最後の名前)が答え。
#           項数も 2,3,4,3 と混ぜ、特定の条件だけが有利にならないようにする。
# N_SHOT_ITEMS = [] にすれば手本なしに戻せる。

# (項数, どちらが答えか)。項数 1 は単項述語(rel0 の形)。
# 比較級の例だけだと rel0 が書式を教わらず不利になるため、単項の例も入れる。
# 答えは「最初」3問・「最後」3問で釣り合わせる。
N_SHOT_ITEMS = [
    (1, "hi"),   # 単項 / 答えは最初
    (2, "hi"),   # 2項   / 答えは最初
    (3, "lo"),   # 3項   / 答えは最後
    (1, "lo"),   # 単項 / 答えは最後
    (4, "hi"),   # 4項   / 答えは最初
    (3, "lo"),   # 3項   / 答えは最後
]


def build_prefix():
    """全条件で共通の手本を1つだけ作る(引数を取らない = 条件によって変わらない)。"""
    if not N_SHOT_ITEMS:
        return ""
    rng = random.Random(1234)
    lines = []
    for n_terms, which in N_SHOT_ITEMS:
        comp, sup_hi, sup_lo, pos_hi, pos_lo = rng.choice(DIMS)

        if n_terms == 1:
            # 単項述語(統合不要)の形。rel0 と同じ書式
            a, b = rng.sample(NAMES, 2)
            prem = f"{a} is {pos_hi}. {b} is {pos_lo}."
            if which == "hi":
                lines.append(f"{prem} The {pos_hi} one is {a}.")
            else:
                lines.append(f"{prem} The {pos_lo} one is {b}.")
            continue

        people = rng.sample(NAMES, n_terms)
        prem = " ".join(f"{people[j]} is {comp} than {people[j+1]}."
                        for j in range(n_terms - 1))
        if which == "hi":
            lines.append(f"{prem} The {sup_hi} one is {people[0]}.")
        else:
            lines.append(f"{prem} The {sup_lo} one is {people[-1]}.")
    return " ".join(lines) + " "


PREFIX = build_prefix()      # 1度だけ作り、全条件で使い回す
print("共通の手本:")
print(" ", PREFIX.strip())
print()

def make_nterm(n_terms, n):
    """N項推移推論。

    設計:
      - 質問語は固定(最上位を問う)。読み出しではなく【順序の組み立て】を差分に乗せるため
      - corrupted は先頭要素の名前を1語だけ差し替える -> 順序の頂点が変わる
      - 前提の提示順はランダム -> 位置だけで答えを当てる戦略を潰す
    """
    out = []
    for _ in range(n):
        people = random.sample(NAMES, n_terms + 1)   # 1つ多く取り、最後を差し替え用に使う
        chain = people[:n_terms]                     # clean の順序 (chain[0] が最上位)
        alt = people[n_terms]                        # corrupted で chain[0] と入れ替える名前
        comp, sup_hi, sup_lo, pos_hi, pos_lo = random.choice(DIMS)

        # 前提を作り、提示順をシャッフルする
        idx = list(range(n_terms - 1))
        random.shuffle(idx)

        def build(top):
            c2 = [top] + chain[1:]
            prem = [f"{c2[j]} is {comp} than {c2[j+1]}." for j in range(n_terms - 1)]
            return " ".join(prem[j] for j in idx)

        out.append(dict(
            clean     = f"{PREFIX}{build(chain[0])} The {sup_hi} one is",
            corrupted = f"{PREFIX}{build(alt)} The {sup_hi} one is",
            correct   = f" {chain[0]}",
            wrong     = f" {alt}"))
    return out


def make_rel0(n):
    """関係0個(単項述語): 統合が不要な対照群。

    Halford の定義では単項関係 = 単一の引数に対する述語(クラス所属)。
    質問語は固定し、corrupted では述語の付き先を1語だけ差し替える。
    文の構造は rel1 以降と揃えてある。
    """
    out = []
    for _ in range(n):
        a, b, alt = random.sample(NAMES, 3)
        comp, sup_hi, sup_lo, pos_hi, pos_lo = random.choice(DIMS)
        order = [f"{{0}} is {pos_hi}.", f"{b} is {pos_lo}."]
        random.shuffle(order)
        body = " ".join(order)
        out.append(dict(
            clean     = f"{PREFIX}{body.format(a)} The {pos_hi} one is",
            corrupted = f"{PREFIX}{body.format(alt)} The {pos_hi} one is",
            correct   = f" {a}",
            wrong     = f" {alt}"))
    return out


def uniform_length(items):
    """条件内で最も多い長さに揃える。揃っていないとバッチにパディングが入る。"""
    from collections import Counter
    lens = [model.to_tokens(t["clean"]).shape[1] for t in items]
    target = Counter(lens).most_common(1)[0][0]
    kept = [t for t, L in zip(items, lens) if L == target]
    return kept, target


TASK_SETS = {}
for name, gen in [("rel0", lambda: make_rel0(N_PER_COND)),
                  ("rel1", lambda: make_nterm(2, N_PER_COND)),
                  ("rel2", lambda: make_nterm(3, N_PER_COND)),
                  ("rel3", lambda: make_nterm(4, N_PER_COND)),
                  ("rel4", lambda: make_nterm(5, N_PER_COND))]:
    items, L = uniform_length(gen())
    TASK_SETS[name] = items
    print(f"{name}  {len(items):>3} 問  (系列長 {L} に統一)")

FAMILY = "rel2"     # <<< 分析する条件。変えたら STEP4 以降を再実行する

print()
print("合計", sum(len(v) for v in TASK_SETS.values()), "問  / 選択中:", FAMILY)
print()
for fam in TASK_SETS:
    t = TASK_SETS[fam][0]
    print(f"[{fam}]")
    print("  clean    :", t["clean"])
    print("  corrupted:", t["corrupted"])
    print("  答え     :", repr(t["correct"]), "/", repr(t["wrong"]))

使える名前 28 / 60
使える比較語 6 / 8  -> ['taller', 'older', 'richer', 'stronger', 'bigger', 'heavier']

共通の手本:
  Fred is strong. Ann is weak. The strong one is Fred. Peter is taller than Nick. The tallest one is Peter. Grace is taller than Henry. Henry is taller than Dan. The shortest one is Dan. Laura is tall. Jane is short. The short one is Jane. Ann is older than Peter. Peter is older than Jane. Jane is older than Frank. The oldest one is Ann. Amy is bigger than Adam. Adam is bigger than Tom. The smallest one is Tom.

rel0   50 問  (系列長 112 に統一)


rel1   42 問  (系列長 110 に統一)
rel2   43 問  (系列長 116 に統一)
rel3   40 問  (系列長 122 に統一)
rel4   37 問  (系列長 128 に統一)

合計 212 問  / 選択中: rel2

[rel0]
  clean    : Fred is strong. Ann is weak. The strong one is Fred. Peter is taller than Nick. The tallest one is Peter. Grace is taller than Henry. Henry is taller than Dan. The shortest one is Dan. Laura is tall. Jane is short. The short one is Jane. Ann is older than Peter. Peter is older than Jane. Jane is older than Frank. The oldest one is Ann. Amy is bigger than Adam. Adam is bigger than Tom. The smallest one is Tom. Mark is weak. Tony is strong. The strong one is
  corrupted: Fred is strong. Ann is weak. The strong one is Fred. Peter is taller than Nick. The tallest one is Peter. Grace is taller than Henry. Henry is taller than Dan. The shortest one is Dan. Laura is tall. Jane is short. The short one is Jane. Ann is older than Peter. Peter is older than Jane. Jane is older than Frank. The oldest one is Ann. Amy is bigger than Adam. Adam is big

In [3]:
# ========== STEP 4 / 12 : 検査 → 展開 → スクリーニング ==========
# ---- 検証 → 公式デモの変数名へ展開 → スクリーニング ----

def validate_family(items, name=""):
    ok, ng = [], []
    for k, t in enumerate(items):
        ct = model.to_tokens(t["clean"])
        pt = model.to_tokens(t["corrupted"])
        c_ids = model.to_tokens(t["correct"], prepend_bos=False)[0]
        w_ids = model.to_tokens(t["wrong"],   prepend_bos=False)[0]
        reasons = []
        if ct.shape[1] != pt.shape[1]:
            reasons.append(f"トークン長不一致 clean={ct.shape[1]} corrupted={pt.shape[1]}")
        if c_ids[0].item() == w_ids[0].item():
            reasons.append("correct と wrong の先頭トークンが同一")
        if ct.shape[1] == pt.shape[1]:
            nd = (ct != pt).sum().item()
            if nd == 0:
                reasons.append("clean と corrupted が同一")
            elif nd > 1:
                reasons.append(f"変更箇所が {nd} 個所(1個所推奨)")
        (ng if reasons else ok).append((k, reasons) if reasons else t)
    print(f"[{name}] 採用 {len(ok)} / 除外 {len(ng)}")
    for k, r in ng:
        print(f"   除外 #{k}: {'; '.join(r)}")
    return ok


TASK_SETS_OK = {}
for fam, its in TASK_SETS.items():
    if not its:
        continue
    kept = validate_family(its, fam)
    if kept:
        TASK_SETS_OK[fam] = kept
print("分析可能な族:", list(TASK_SETS_OK.keys()))
print()

# ---- 公式デモが使う変数名に詰める ----
# パッチングは全活性をキャッシュするのでメモリを食う。
# スクリーニングは全問で行い、パッチングに回すのは先頭 N_PATCH 問に絞る。
# OOM が出たらこの数を減らす。
N_PATCH = 8

items_all = TASK_SETS_OK[FAMILY]
items = items_all[:N_PATCH]
print(f"パッチング対象: {len(items)} 問 (スクリーニングは {len(items_all)} 問すべて)")

clean_tokens     = model.to_tokens([t["clean"]     for t in items])
corrupted_tokens = model.to_tokens([t["corrupted"] for t in items])
answer_token_indices = torch.tensor(
    [[model.to_tokens(t["correct"], prepend_bos=False)[0, 0].item(),
      model.to_tokens(t["wrong"],   prepend_bos=False)[0, 0].item()] for t in items],
    device=device)

prompts       = [t["clean"] for t in items]          # Exploratory_Analysis_Demo が使う
answers       = [(t["correct"], t["wrong"]) for t in items]
answer_tokens = answer_token_indices
tokens        = clean_tokens

print(f"族: {FAMILY} / {len(items)} 事例 / 系列長 {clean_tokens.shape[1]}")
print("相違位置:", (clean_tokens[0] != corrupted_tokens[0]).nonzero().flatten().tolist())
print(model.to_str_tokens(clean_tokens[0]))
print()


# ---- パディング検出(バッチ内で長さが揃っているか) ----
# 揃っていないと右側に詰め物が入り、最終位置が実トークンでなくなって測定が壊れる。
_lens = {model.to_tokens(t["clean"]).shape[1] for t in items}
if len(_lens) > 1:
    raise RuntimeError(f"バッチ内でトークン長が不揃い: {sorted(_lens)}  -> パディングが入り測定が壊れます")
_last = model.to_str_tokens(clean_tokens[0])[-1]
print("最終トークン:", repr(_last), "(<|endoftext|> ならパディング混入)")

# ---- スクリーニング: そもそも解けているか ----
with torch.no_grad():
    _cl = model(clean_tokens)
    _clt = _cl[:, -1].argmax(-1)
    del _cl
    _co = model(corrupted_tokens)
    _cot = _co[:, -1].argmax(-1)
    del _co
acc_clean     = (_clt == answer_token_indices[:, 0]).float().mean().item()
acc_corrupted = (_cot == answer_token_indices[:, 1]).float().mean().item()
print(f"clean 正答率 {acc_clean:.1%} / corrupted 正答率 {acc_corrupted:.1%}")
if min(acc_clean, acc_corrupted) < 0.5:
    print("警告: 正答率が低い。この族はパッチングにかけても解釈できない可能性が高い。")

# ---- 全条件をまとめて診断 ----
# 注意: 出力ロジットは [問題数, 系列長, 語彙数] で、語彙が15万あるため巨大になる。
#       まとめて流すと数GBのテンソルができて後の処理が落ちる。
#       小分けにして、必要な最終位置だけ取り出し、都度解放する。
import gc

CHUNK = 8

def _argmax_last(prompts_list):
    """最終位置の最尤トークンだけを返す(巨大ロジットを溜めない)"""
    outs = []
    for s in range(0, len(prompts_list), CHUNK):
        toks = model.to_tokens(prompts_list[s:s + CHUNK])
        with torch.no_grad():
            lg = model(toks)
            outs.append(lg[:, -1].argmax(-1).cpu())
        del lg, toks
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return torch.cat(outs)

print()
print(f"{'条件':<6}{'clean':>8}{'corrupt':>9}{'直近を答えた率':>16}")
print("-" * 42)
for _fam, _items in TASK_SETS.items():
    _ai = torch.tensor(
        [[model.to_tokens(t["correct"], prepend_bos=False)[0, 0].item(),
          model.to_tokens(t["wrong"],   prepend_bos=False)[0, 0].item()] for t in _items])
    _a = _argmax_last([t["clean"] for t in _items])
    _b = _argmax_last([t["corrupted"] for t in _items])
    _acc_c = (_a == _ai[:, 0]).float().mean().item()
    _acc_p = (_b == _ai[:, 1]).float().mean().item()
    _rec   = (_a == _ai[:, 1]).float().mean().item()
    print(f"{_fam:<6}{_acc_c:>8.1%}{_acc_p:>9.1%}{_rec:>16.1%}")
    del _a, _b, _ai

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print()
    print(f"診断後の使用VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print()
print("直近を答えた率が高い = 推移律を解かずコピーしているだけ")


[rel0] 採用 50 / 除外 0
[rel1] 採用 42 / 除外 0
[rel2] 採用 43 / 除外 0
[rel3] 採用 40 / 除外 0
[rel4] 採用 37 / 除外 0
分析可能な族: ['rel0', 'rel1', 'rel2', 'rel3', 'rel4']

パッチング対象: 8 問 (スクリーニングは 43 問すべて)
族: rel2 / 8 事例 / 系列長 116
相違位置: [100]
['Fred', ' is', ' strong', '.', ' Ann', ' is', ' weak', '.', ' The', ' strong', ' one', ' is', ' Fred', '.', ' Peter', ' is', ' taller', ' than', ' Nick', '.', ' The', ' tallest', ' one', ' is', ' Peter', '.', ' Grace', ' is', ' taller', ' than', ' Henry', '.', ' Henry', ' is', ' taller', ' than', ' Dan', '.', ' The', ' shortest', ' one', ' is', ' Dan', '.', ' Laura', ' is', ' tall', '.', ' Jane', ' is', ' short', '.', ' The', ' short', ' one', ' is', ' Jane', '.', ' Ann', ' is', ' older', ' than', ' Peter', '.', ' Peter', ' is', ' older', ' than', ' Jane', '.', ' Jane', ' is', ' older', ' than', ' Frank', '.', ' The', ' oldest', ' one', ' is', ' Ann', '.', ' Amy', ' is', ' bigger', ' than', ' Adam', '.', ' Adam', ' is', ' bigger', ' than', ' Tom', '.', ' The', ' smalles

---
## Phase 2: 反応度マップ

指標は公式の**正規化ロジット差**（corrupted=0 / clean=1）。
正規化されているので族をまたいで図を比較できる — Phase 3 の交差の前提。

図は **8枚**（ブロック別3枚 + ヘッド別5枚）。

In [11]:
# ========== STEP 5 / 12 : 指標の定義（正規化ロジット差） ==========
# ---- [公式流用] Activation_Patching_in_TL_Demo cell 16 ----
def get_logit_diff(logits, answer_token_indices=answer_token_indices):
    if len(logits.shape)==3:
        # Get final logits only
        logits = logits[:, -1, :]
    correct_logits = logits.gather(1, answer_token_indices[:, 0].unsqueeze(1))
    incorrect_logits = logits.gather(1, answer_token_indices[:, 1].unsqueeze(1))
    return (correct_logits - incorrect_logits).mean()

clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

clean_logit_diff = get_logit_diff(clean_logits, answer_token_indices).item()
print(f"Clean logit diff: {clean_logit_diff:.4f}")

corrupted_logit_diff = get_logit_diff(corrupted_logits, answer_token_indices).item()
print(f"Corrupted logit diff: {corrupted_logit_diff:.4f}")

# ---- [公式流用] Activation_Patching_in_TL_Demo cell 17 ----
CLEAN_BASELINE = clean_logit_diff
CORRUPTED_BASELINE = corrupted_logit_diff
def ioi_metric(logits, answer_token_indices=answer_token_indices):
    return (get_logit_diff(logits, answer_token_indices) - CORRUPTED_BASELINE) / (CLEAN_BASELINE  - CORRUPTED_BASELINE)

print(f"Clean Baseline is 1: {ioi_metric(clean_logits).item():.4f}")
print(f"Corrupted Baseline is 0: {ioi_metric(corrupted_logits).item():.4f}")

# ---- 副作用の分離 ----
# ロジット差は「正解を上げた」と「不正解を下げた」を区別しない。
# 不正解を下げただけの部品を推論部品と誤認しないよう両者を分けて記録する。
def logit_parts(logits, answer_token_indices=answer_token_indices):
    if len(logits.shape) == 3:
        logits = logits[:, -1, :]
    c = logits.gather(1, answer_token_indices[:, 0].unsqueeze(1)).mean().item()
    w = logits.gather(1, answer_token_indices[:, 1].unsqueeze(1)).mean().item()
    return round(c, 3), round(w, 3)

print("clean     (正解, 不正解):", logit_parts(clean_logits))
print("corrupted (正解, 不正解):", logit_parts(corrupted_logits))

Clean logit diff: 2.1632
Corrupted logit diff: -2.6555
Clean Baseline is 1: 1.0000
Corrupted Baseline is 0: 0.0000
clean     (正解, 不正解): (20.356, 18.193)
corrupted (正解, 不正解): (17.502, 20.157)


In [5]:
# ========== STEP 6 / 12 : 図8枚 — denoising（十分性） ==========
# ============ 方向1: denoising (十分性) ============
# corrupted を走らせながら clean の活性を1つ差し込む。
# 明るい = その部品だけで答えが決まる(十分)

# ---- [公式流用] Activation_Patching_in_TL_Demo cell 28  (ブロック別 3枚) ----
# NBVAL_SKIP
# Heavy patching computation — too slow for CI without GPU
every_block_result = patching.get_act_patch_block_every(model, corrupted_tokens, clean_cache, ioi_metric)
imshow(every_block_result, facet_col=0, facet_labels=["Residual Stream", "Attn Output", "MLP Output"], title="Activation Patching Per Block (denoising = 十分性)", xaxis="Position", yaxis="Layer", zmax=1, zmin=-1, x= [f"{tok}_{i}" for i, tok in enumerate(model.to_str_tokens(clean_tokens[0]))])

# ---- [公式流用] Activation_Patching_in_TL_Demo cell 30  (ヘッド別 5枚) ----
# NBVAL_SKIP
every_head_all_pos_act_patch_result = patching.get_act_patch_attn_head_all_pos_every(model, corrupted_tokens, clean_cache, ioi_metric)
imshow(every_head_all_pos_act_patch_result, facet_col=0, facet_labels=["Output", "Query", "Key", "Value", "Pattern"], title="Activation Patching Per Head (All Pos, denoising = 十分性)", xaxis="Head", yaxis="Layer", zmax=1, zmin=-1)
# [markdown]
# We can also do by head *and* by position. This is a bit slow, but it can give useful + fine-grained detail

  0%|          | 0/2112 [00:00<?, ?it/s]

  0%|          | 0/2112 [00:00<?, ?it/s]

KeyboardInterrupt: 

### 方向2: noising（必要性）— 狙った部分を壊す

さきほどと**引数を入れ替えるだけ**。公式関数はそのまま使う。

| | 実行するトークン | 差し込む活性 | 測るもの |
|---|---|---|---|
| **denoising** | corrupted | clean | **十分性** … その部品だけで答えが決まるか |
| **noising** | clean | corrupted | **必要性** … その部品を壊すと答えが崩れるか |

**この2つは非対称で、一方が他方を含意しない。**
AND型の回路（複数部品が揃って初めて働く）は denoising では見えにくく、
OR型の回路（どれか1つで足りる）は noising では見えにくい。だから両方出して突き合わせる。

生の指標は「clean のまま = 1」なので、**壊れたほど 0 に近づく**。
図として見やすいよう `damage = 1 - スコア` に直したものも併せて描く（**明るい = よく壊れた = 必要**）。

In [ ]:
# ========== STEP 7 / 12 : 図8枚 — noising（必要性・狙って壊す） ==========
# ============ 方向2: noising (必要性) ============
# 公式関数の引数を入れ替えるだけ: clean を走らせながら corrupted の活性を差し込む。
# 生スコア  … 1 に近い = 壊れていない(不要) / 0 に近い = 壊れた(必要)
# damage    … 1 - 生スコア。明るい = よく壊れた = 必要

every_block_noise = patching.get_act_patch_block_every(
    model, clean_tokens, corrupted_cache, ioi_metric)

imshow(1 - every_block_noise,
       facet_col=0, facet_labels=["Residual Stream", "Attn Output", "MLP Output"],
       title="Noising damage per Block (明るい = 壊すと崩れる = 必要)",
       xaxis="Position", yaxis="Layer", zmax=1, zmin=-1,
       x=[f"{tok}_{i}" for i, tok in enumerate(model.to_str_tokens(clean_tokens[0]))])

every_head_all_pos_noise = patching.get_act_patch_attn_head_all_pos_every(
    model, clean_tokens, corrupted_cache, ioi_metric)

imshow(1 - every_head_all_pos_noise,
       facet_col=0, facet_labels=["Output", "Query", "Key", "Value", "Pattern"],
       title="Noising damage per Head (明るい = 壊すと崩れる = 必要)",
       xaxis="Head", yaxis="Layer", zmax=1, zmin=-1)

### 両方向の突き合わせ

十分でも必要でもある部品が、回路の中核である可能性が高い。
片方にしか出ない部品は、AND / OR の構造を示唆する。

In [ ]:
# ========== STEP 8 / 12 : 十分性 vs 必要性 の突き合わせ ==========
# denoising と noising を散布図で突き合わせる
ACT_CMP = 0   # 0=Output

_d = every_head_all_pos_act_patch_result[ACT_CMP].flatten()          # 十分性
_n = (1 - every_head_all_pos_noise)[ACT_CMP].flatten()               # 必要性(damage)

scatter(x=_d, y=_n, hover_name=head_labels,
        xaxis="denoising (十分性)", yaxis="noising damage (必要性)",
        title="十分性 vs 必要性 — 右上が回路の中核候補")

_score = _d + _n
_top = _score.argsort(descending=True)[:10]
print("両方向あわせた上位10ヘッド")
for j in _top.tolist():
    print(f"  {head_labels[j]:>8}   十分性 {_d[j]:+.3f}   必要性 {_n[j]:+.3f}")

---
## Phase 3: 直接寄与と交差

まず公式の直接寄与分析（DLA）。パッチングとは別経路で同じ部品が浮かぶかを確認する。
あわせて `head_detector` で、候補が既知の型（前トークン / 重複トークン / 誘導）に当たるか判定する。

**既知の型に当たったら要注意** — 推論回路ではなくコピー機構を見ている可能性が高い。

In [ ]:
# ========== STEP 9 / 12 : DLA（直接寄与）+ 既知ヘッド型の検出 ==========
# ---- 必要なもの以外を全部解放してから始める ----
# run_with_cache は全層の活性を保持するため、前の STEP のキャッシュが残っていると OOM になる。
# モデルと入力だけ残して、他は捨てる。
free_except("model", "tokens", "clean_tokens", "corrupted_tokens",
            "answer_tokens", "answer_token_indices", "prompts", "answers",
            "items", "items_all", "TASK_SETS", "TASK_SETS_OK", "NAMES", "DIMS",
            "head_labels", "ALL_HEAD_LABELS", "imshow", "line", "scatter",
            "FAMILY", "PREFIX", "device", "model_path")

# ---- キャッシュ取得 ----
original_logits, cache = model.run_with_cache(tokens)

# ---- [公式流用] Exploratory_Analysis_Demo cell 28 ----
def logits_to_ave_logit_diff(logits, answer_tokens, per_prompt=False):
    # Only the final logits are relevant for the answer
    final_logits = logits[:, -1, :]
    answer_logits = final_logits.gather(dim=-1, index=answer_tokens)
    answer_logit_diff = answer_logits[:, 0] - answer_logits[:, 1]
    if per_prompt:
        return answer_logit_diff
    else:
        return answer_logit_diff.mean()


print(
    "Per prompt logit difference:",
    logits_to_ave_logit_diff(original_logits, answer_tokens, per_prompt=True)
    .detach()
    .cpu()
    .round(decimals=3),
)
original_average_logit_diff = logits_to_ave_logit_diff(original_logits, answer_tokens)
print(
    "Average logit difference:",
    round(logits_to_ave_logit_diff(original_logits, answer_tokens).item(), 3),
)

# ---- [公式流用] Exploratory_Analysis_Demo cell 34 ----
# TransformerBridge doesn't have tokens_to_residual_directions yet,
# so we implement it inline using model.unembed.W_U
W_U = model.unembed.W_U  # [d_model, d_vocab]
answer_residual_directions = W_U[:, answer_tokens]
answer_residual_directions = einops.rearrange(
    answer_residual_directions, "d_model batch correct_incorrect -> batch correct_incorrect d_model"
)
print("Answer residual directions shape:", answer_residual_directions.shape)
logit_diff_directions = (
    answer_residual_directions[:, 0] - answer_residual_directions[:, 1]
)
print("Logit difference directions shape:", logit_diff_directions.shape)

# ---- [公式流用] Exploratory_Analysis_Demo cell 39 ----
def residual_stack_to_logit_diff(
    residual_stack: Float[torch.Tensor, "components batch d_model"],
    cache: ActivationCache,
) -> float:
    scaled_residual_stack = cache.apply_ln_to_stack(
        residual_stack, layer=-1, pos_slice=-1
    )
    return einsum(
        "... batch d_model, batch d_model -> ...",
        scaled_residual_stack,
        logit_diff_directions,
    ) / len(prompts)

# ---- [公式流用] Exploratory_Analysis_Demo cell 41 (層ごと・累積) ----
accumulated_residual, labels = cache.accumulated_resid(
    layer=-1, incl_mid=True, pos_slice=-1, return_labels=True
)
logit_lens_logit_diffs = residual_stack_to_logit_diff(accumulated_residual, cache)
line(
    logit_lens_logit_diffs,
    x=np.arange(model.cfg.n_layers * 2 + 1) / 2,
    hover_name=labels,
    title="Logit Difference From Accumulate Residual Stream",
)

# ---- [公式流用] Exploratory_Analysis_Demo cell 44 (層ごと・成分別) ----
per_layer_residual, labels = cache.decompose_resid(
    layer=-1, pos_slice=-1, return_labels=True
)
per_layer_logit_diffs = residual_stack_to_logit_diff(per_layer_residual, cache)
line(per_layer_logit_diffs, hover_name=labels, title="Logit Difference From Each Layer")

# ---- [公式流用] Exploratory_Analysis_Demo cell 47 ----
per_head_residual, labels = cache.stack_head_results(
    layer=-1, pos_slice=-1, return_labels=True
)
per_head_logit_diffs = residual_stack_to_logit_diff(per_head_residual, cache)
per_head_logit_diffs = einops.rearrange(
    per_head_logit_diffs,
    "(layer head_index) -> layer head_index",
    layer=model.cfg.n_layers,
    head_index=model.cfg.n_heads,
)
imshow(
    per_head_logit_diffs,
    labels={"x": "Head", "y": "Layer"},
    title="Logit Difference From Each Head",
)

# ---- 既知ヘッド型の検出 ----
# detect_head は文字列(またはその配列)を取る。トークンのテンソルは渡せない。
from transformer_lens.head_detector import detect_head

for head_name in ["previous_token_head", "duplicate_token_head", "induction_head"]:
    scores = detect_head(model, prompts[0], head_name)
    imshow(scores, yaxis="Layer", xaxis="Head", title=f"{head_name} 検出スコア")


# =========================================================
#  数値サマリ（図の中身を表で読む）
# =========================================================
# 図はホバーしないとラベルが出ないので、同じ数値を文字で出す。

# ---- 層ごと（Attn / MLP 別）----
_res, _lab = cache.decompose_resid(layer=-1, pos_slice=-1, return_labels=True)
_vals = residual_stack_to_logit_diff(_res, cache).detach().cpu()

_pairs = [(_lab[i], float(_vals[i])) for i in range(len(_lab))]
_srt = sorted(_pairs, key=lambda x: -x[1])

print("=" * 46)
print("層ごとの寄与 — 上位10（答えを正解に押した）")
print("=" * 46)
for name, v in _srt[:10]:
    print(f"  {name:<16}{v:+.3f}")

print()
print("層ごとの寄与 — 下位5（答えを不正解に押した）")
for name, v in _srt[-5:]:
    print(f"  {name:<16}{v:+.3f}")

# Attn と MLP のどちらが効いているかの総計
_attn = sum(v for n, v in _pairs if "attn" in n)
_mlp = sum(v for n, v in _pairs if "mlp" in n)
print()
print(f"  Attn 合計: {_attn:+.3f}   MLP 合計: {_mlp:+.3f}")
print(f"  -> {'Attn が主役' if abs(_attn) > abs(_mlp) else 'MLP が主役'}")

# ---- ヘッドごと ----
_hd = per_head_logit_diffs.detach().cpu()
_flat = [(l, h, float(_hd[l, h])) for l in range(_hd.shape[0]) for h in range(_hd.shape[1])]
_hsrt = sorted(_flat, key=lambda x: -x[2])

print()
print("=" * 46)
print("ヘッドごとの寄与 — 上位10")
print("=" * 46)
for l, h, v in _hsrt[:10]:
    print(f"  L{l:<2}H{h:<2}  {v:+.3f}")

print()
print("ヘッドごとの寄与 — 下位5（逆方向に押している）")
for l, h, v in _hsrt[-5:]:
    print(f"  L{l:<2}H{h:<2}  {v:+.3f}")

# ---- 上位ヘッドが既知の型かどうか ----
print()
print("=" * 46)
print("上位5ヘッドが既知の型に当たるか（高い=コピー機構の疑い）")
print("=" * 46)
_names = ["previous_token_head", "duplicate_token_head", "induction_head"]
_scores = {n: detect_head(model, prompts[0], n) for n in _names}
print(f"  {'ヘッド':<10}" + "".join(f"{n.replace('_head',''):>18}" for n in _names))
for l, h, v in _hsrt[:5]:
    row = "".join(f"{float(_scores[n][l, h]):>18.3f}" for n in _names)
    print(f"  L{l:<2}H{h:<7}" + row)

print()
print("次の一手: 上位ヘッドのうち既知の型に当たらないものが、推論固有の部品の候補。")


### 族をまたいだ交差 — ここが本実験の核心

`FAMILY` を変えて Phase 1〜3 を再実行し、族ごとに下のセルで結果を溜める。
2族以上溜まると交差が計算される。

In [ ]:
# ========== STEP 10 / 12 : 結果の保存 + 族をまたいだ交差  ★核心★ ==========
# ---- 現在の族の結果を保存し、2族以上あれば交差を取る ----
try:
    RESULTS
except NameError:
    RESULTS = {}

RESULTS[FAMILY] = dict(
    head_denoise = every_head_all_pos_act_patch_result.detach().cpu(),   # [5, layer, head] 十分性
    head_noise   = (1 - every_head_all_pos_noise).detach().cpu(),        # [5, layer, head] 必要性(damage)
    block_denoise= every_block_result.detach().cpu(),                    # [3, layer, pos]
    block_noise  = (1 - every_block_noise).detach().cpu(),               # [3, layer, pos]
    head_dla     = per_head_logit_diffs.detach().cpu(),                  # [layer, head]
    acc_clean    = acc_clean,
    n_items      = len(items),
)
print("保存:", FAMILY, "| 溜まっている族:", list(RESULTS.keys()))
print()

ACT_TYPES = ["Output", "Query", "Key", "Value", "Pattern"]
ACT       = 0              # 0=Output。必要なら 1..4
TOPK      = 15             # 各族で上位いくつを「効いている」とみなすか
DIRECTION = "head_noise"   # "head_noise"(必要性) か "head_denoise"(十分性)

fams = list(RESULTS.keys())
if len(fams) < 2:
    print(f"交差にはあと {2 - len(fams)} 族必要です(FAMILY を変えて再実行)")
else:
    masks = {}
    for f in fams:
        m = RESULTS[f][DIRECTION][ACT]
        th = m.flatten().abs().sort(descending=True).values[TOPK - 1]
        masks[f] = (m.abs() >= th)

    inter = masks[fams[0]].clone()
    for f in fams[1:]:
        inter &= masks[f]
    n_fam_hit = sum(masks[f].int() for f in fams)

    print(f"方向 {DIRECTION} / 指標 {ACT_TYPES[ACT]} / 各族の上位 {TOPK} 件")
    print(f"全 {len(fams)} 族すべてで効いたヘッド: {int(inter.sum())} 件")
    for l, h in inter.nonzero().tolist():
        vals = "  ".join(f"{f}={RESULTS[f][DIRECTION][ACT][l, h]:+.3f}" for f in fams)
        print(f"   L{l}H{h}   {vals}")

    imshow(n_fam_hit.float(), yaxis="Layer", xaxis="Head",
           title=f"何族で効いたか ({DIRECTION}, {ACT_TYPES[ACT]}, 上位{TOPK})")

---
## Phase 4: アブレーションと反復的な絞り込み

1. 候補をゼロ化する
2. 性能が落ちるか測る
3. **削除前後のヘッド別寄与を比較** ← 自己修復（Hydra効果）の検出
4. 候補を絞って 1 に戻る

**自己修復に注意**: 削っても性能が落ちないことがある。予備部品が肩代わりするため。
散布図で寄与が増えたヘッドがあれば、それが予備部品。**まとめて削る必要がある。**

In [5]:
# ========== STEP 11 / 12 : アブレーション（1個削る + 前後比較） ==========
# ---- [公式流用] Exploratory_Analysis_Demo cell 115  (最大寄与ヘッドを1つ削る) ----
top_name_mover = per_head_logit_diffs.flatten().argmax().item()
top_name_mover_layer = top_name_mover // model.cfg.n_heads
top_name_mover_head = top_name_mover % model.cfg.n_heads
print(f"Top Name Mover to ablate: L{top_name_mover_layer}H{top_name_mover_head}")


def ablate_top_head_hook(z: Float[torch.Tensor, "batch pos head_index d_head"], hook):
    z = z.clone()
    z[:, -1, top_name_mover_head, :] = 0
    return z


# Adds a hook into global model state
model.blocks[top_name_mover_layer].attn.hook_z.add_hook(ablate_top_head_hook)
# Runs the model, temporarily adds caching hooks and then removes *all* hooks after running, including the ablation hook.
ablated_logits, ablated_cache = model.run_with_cache(tokens)
print(f"Original logit diff: {original_average_logit_diff:.2f}")
print(
    f"Post ablation logit diff: {logits_to_ave_logit_diff(ablated_logits, answer_tokens).item():.2f}"
)
print(
    f"Direct Logit Attribution of top name mover head: {per_head_logit_diffs.flatten()[top_name_mover].item():.2f}"
)
print(
    f"Naive prediction of post ablation logit diff: {original_average_logit_diff - per_head_logit_diffs.flatten()[top_name_mover].item():.2f}"
)

# ---- [公式流用] Exploratory_Analysis_Demo cell 117  (削除前後の比較) ----
per_head_ablated_residual, labels = ablated_cache.stack_head_results(
    layer=-1, pos_slice=-1, return_labels=True
)
per_head_ablated_logit_diffs = residual_stack_to_logit_diff(
    per_head_ablated_residual, ablated_cache
)
per_head_ablated_logit_diffs = per_head_ablated_logit_diffs.reshape(
    model.cfg.n_layers, model.cfg.n_heads
)
imshow(per_head_ablated_logit_diffs, labels={"x": "Head", "y": "Layer"})
scatter(
    y=per_head_logit_diffs.flatten(),
    x=per_head_ablated_logit_diffs.flatten(),
    hover_name=head_labels,
    range_x=(-3, 3),
    range_y=(-3, 3),
    xaxis="Ablated",
    yaxis="Original",
    title="Original vs Post-Ablation Direct Logit Attribution of Heads",
)

NameError: name 'per_head_logit_diffs' is not defined

In [ ]:
# ========== STEP 12 / 12 : アブレーション（まとめて削る） ==========
# ---- 複数ヘッドの同時アブレーション(自己修復を潰すため) ----
HEADS_TO_ABLATE = [
    # (layer, head),   <-- Phase 3 の交差で出た候補をここに並べる
]

def make_ablation_hook(head_index):
    def hook(z, hook):
        z = z.clone()
        z[:, :, head_index, :] = 0
        return z
    return hook

if not HEADS_TO_ABLATE:
    print("HEADS_TO_ABLATE が空です。Phase 3 の交差の結果を入れてください。")
else:
    model.reset_hooks()
    for layer, head in HEADS_TO_ABLATE:
        model.blocks[layer].attn.hook_z.add_hook(make_ablation_hook(head))
    multi_ablated_logits = model(tokens)
    model.reset_hooks()

    before = logits_to_ave_logit_diff(original_logits, answer_tokens).item()
    after  = logits_to_ave_logit_diff(multi_ablated_logits, answer_tokens).item()
    print(f"削除: {HEADS_TO_ABLATE}")
    print(f"削除前 {before:.3f} -> 削除後 {after:.3f}   (低下 {before - after:.3f} / {(1 - after / before) * 100:.0f}% 減)")

---
## 主張できること・できないこと

**できる** — テストしたテンプレート分布において、ある部品群が複数の推論族で因果的に必要であること

**できない**
- 得られた回路が**最小**であること（十分な部品集合が得られるだけ）
- テストした分布の**外**でも同じであること
- その部品が推論**だけ**を担っていること（1部品が複数機能を持ちうる）

In [4]:
# ========== STEP 13 : 1問ごと・1層ごとの影響度表（順伝播1回） ==========
# 公式 API を使う:
#   ActivationCache.get_full_resid_decomposition(expand_neurons=True,
#                                                project_output_onto=...)
#   -> 注意ヘッド 336行 + ニューロン 116,736行 + embed/bias を一括で返す
#      ラベルは "L20H7"(ヘッド) / "L15N3421"(ニューロン) 形式
#      project_output_onto を渡すと巨大な中間テンソルを作らずに済む
#
# ここで測るのは「その成分が、正解と不正解のロジット差にいくら足したか」。
# 正なら正解を後押し、負なら逆方向に押している。

import gc
import re as _re

import einops
import pandas as pd
import torch

free_except("model", "tokens", "clean_tokens", "corrupted_tokens",
            "answer_tokens", "answer_token_indices", "prompts", "answers",
            "items", "items_all", "TASK_SETS", "TASK_SETS_OK", "NAMES", "DIMS",
            "head_labels", "ALL_HEAD_LABELS", "imshow", "line", "scatter",
            "FAMILY", "PREFIX", "device", "model_path")

N_Q = 8            # 何問ぶん記録するか（メモリと相談。まず 8 問）
TOP_N_NEURON = 10  # 表に載せるニューロンの数（全値は CSV に出す）

qs = tokens[:N_Q]
ans = answer_tokens[:N_Q]

# ---- 正解方向のベクトル（公式 Exploratory_Analysis_Demo と同じ作り方）----
W_U = model.unembed.W_U                      # [d_model, d_vocab]
dirs = W_U[:, ans]                           # [d_model, batch, 2]
dirs = einops.rearrange(dirs, "d b c -> b c d")
logit_diff_dirs = dirs[:, 0] - dirs[:, 1]    # [batch, d_model]

# ---- 順伝播1回 ----
with torch.no_grad():
    logits, cache = model.run_with_cache(qs)

# 予測が当たっているかも記録しておく（正解時と不正解時で比べるため）
pred = logits[:, -1].argmax(-1)
is_correct = (pred == ans[:, 0]).cpu()
ld = (logits[:, -1].gather(1, ans[:, 0:1]) - logits[:, -1].gather(1, ans[:, 1:2])).squeeze(1).cpu()
del logits
gc.collect()
torch.cuda.empty_cache()

# ---- 全成分の寄与を一括取得（ヘッド + ニューロン + embed）----
stack, labels = cache.get_full_resid_decomposition(
    layer=-1,
    expand_neurons=True,
    apply_ln=True,
    pos_slice=-1,
    return_labels=True,
    project_output_onto=logit_diff_dirs.T,   # [d_model, batch]
)
# stack: [component, batch, batch] の対角が各問題の値になるので取り出す
if stack.ndim == 3:
    stack = torch.diagonal(stack, dim1=1, dim2=2)   # [component, batch]
attr = stack.float().cpu()                          # [component, batch]
del stack
gc.collect()
torch.cuda.empty_cache()

print(f"成分数 {attr.shape[0]}  /  問題数 {attr.shape[1]}")
print(f"内訳: ヘッド {model.cfg.n_layers * model.cfg.n_heads} + "
      f"ニューロン {model.cfg.n_layers * model.cfg.d_mlp} + その他")
print()

# ---- ラベルを (層, 種別, 番号) に分解 ----
lay, kind, num = [], [], []
for s in labels:
    m = _re.fullmatch(r"L(\d+)H(\d+)", s)
    if m:
        lay.append(int(m.group(1))); kind.append("head"); num.append(int(m.group(2))); continue
    m = _re.fullmatch(r"L(\d+)N(\d+)", s)
    if m:
        lay.append(int(m.group(1))); kind.append("neuron"); num.append(int(m.group(2))); continue
    m = _re.fullmatch(r"(\d+)_mlp_out", s)
    if m:
        lay.append(int(m.group(1))); kind.append("mlp"); num.append(-1); continue
    lay.append(-1); kind.append("other"); num.append(-1)

lay = torch.tensor(lay); kind_arr = kind; num = torch.tensor(num)

# ---- 写真の形で表示（1問 × 1層ごと）----
def show_question(qi, layers=None, top_n=TOP_N_NEURON):
    """1問について、層ごとに Attn ヘッドと FFN ニューロンの影響度を並べる"""
    layers = layers if layers is not None else range(model.cfg.n_layers)
    print("=" * 78)
    print(f"問題 #{qi}   予測={model.to_string(pred[qi:qi+1])!r}  "
          f"正解={'○' if bool(is_correct[qi]) else '×'}  ロジット差={ld[qi]:+.3f}")
    print(f"  {items[qi]['clean'][-70:]}")
    print("=" * 78)
    v = attr[:, qi]
    for L in layers:
        hm = (lay == L) & torch.tensor([k == "head" for k in kind_arr])
        nm = (lay == L) & torch.tensor([k == "neuron" for k in kind_arr])
        hv = v[hm]
        nv = v[nm]
        print(f"[層 {L:2d}]  Attn合計 {hv.sum():+7.3f}   FFN合計 {nv.sum():+7.3f}")
        print("    Attn : " + "  ".join(f"H{i}:{hv[i]:+.3f}" for i in range(len(hv))))
        if len(nv):
            top = nv.abs().argsort(descending=True)[:top_n]
            print(f"    FFN  : " + "  ".join(f"N{int(num[nm][t])}:{nv[t]:+.3f}" for t in top))
        print()

show_question(0, layers=range(model.cfg.n_layers))

# ---- 表（DataFrame）にして CSV へ ----
df = pd.DataFrame({
    "label": labels,
    "layer": lay.tolist(),
    "kind": kind_arr,
    "index": num.tolist(),
})
for qi in range(attr.shape[1]):
    df[f"q{qi}"] = attr[:, qi].tolist()
df["mean"] = attr.mean(dim=1).tolist()
df["consistency"] = (attr > 0).float().mean(dim=1).tolist()   # 正の寄与だった問題の割合

csv_path = f"/content/influence_{FAMILY}.csv"
df.to_csv(csv_path, index=False)
print(f"CSV 出力: {csv_path}  ({len(df)} 行)")
print()

# ---- 上位20成分（平均と一貫性つき）----
top = df.reindex(df["mean"].abs().sort_values(ascending=False).index).head(20)
print("=" * 78)
print("影響の大きい成分 上位20（全問平均）")
print("=" * 78)
print(f"  {'成分':<12}{'種別':<8}{'層':>3}{'平均':>9}{'一貫性':>8}")
for _, r in top.iterrows():
    print(f"  {r['label']:<12}{r['kind']:<8}{int(r['layer']):>3}"
          f"{r['mean']:>9.3f}{r['consistency']:>8.0%}")

print()
print("一貫性 = その成分が正の寄与だった問題の割合。")
print("平均が大きくても一貫性が低ければ、数問の外れ値が作った見かけ上の値。")


解放: 0 個 / GPU 使用中: 2.83 GB
成分数 117074  /  問題数 8
内訳: ヘッド 336 + ニューロン 116736 + その他

問題 #0   予測=' Bob'  正解=○  ロジット差=+6.170
  is stronger than Mark. Mark is stronger than Ann. The strongest one is
[層  0]  Attn合計  -0.007   FFN合計  -0.100
    Attn : H0:+0.001  H1:-0.007  H2:+0.003  H3:-0.003  H4:+0.001  H5:-0.001  H6:-0.002  H7:-0.001  H8:+0.003  H9:+0.002  H10:+0.003  H11:-0.002  H12:+0.001  H13:-0.004
    FFN  : N1375:-0.059  N2628:-0.018  N1138:-0.018  N3113:+0.015  N2640:-0.014  N4283:-0.012  N2919:-0.012  N714:-0.011  N1180:+0.010  N2546:-0.010

[層  1]  Attn合計  +0.096   FFN合計  -0.044
    Attn : H0:+0.032  H1:+0.036  H2:-0.021  H3:+0.005  H4:-0.015  H5:-0.014  H6:-0.019  H7:+0.005  H8:+0.035  H9:-0.014  H10:+0.007  H11:+0.048  H12:-0.009  H13:+0.020
    FFN  : N3890:+0.057  N1942:+0.049  N3480:+0.021  N3672:+0.015  N4475:+0.015  N99:-0.015  N3808:+0.014  N64:-0.011  N3555:+0.011  N1533:+0.011

[層  2]  Attn合計  -0.018   FFN合計  -0.014
    Attn : H0:-0.014  H1:+0.001  H2:+0.000  H3:+0.012  H4

In [5]:
# ========== STEP 14 : Q / K / V の影響度（1問ごと・勾配ベース） ==========
# 出典: demos/Attribution_Patching_Demo.ipynb の attr_patch_head_vector
#       計算式は公式のまま:  影響度 = corrupted の勾配 × (clean の活性 - corrupted の活性)
#
# 公式からの変更点（メモリ対策）:
#   1. 公式は clean と corrupted の両方で逆伝播するが、この式に必要なのは
#      corrupted 側の勾配だけ。clean は勾配なしの順伝播で足りる -> 逆伝播1回に削減
#   2. 公式は全フックをキャッシュする("_input" not in name)。MLP の中間活性
#      (4864次元) や注意パターンまで保存されて溢れるので、q/k/v/z だけに絞る
#   3. 取り出した値は即 CPU に移して GPU を解放する
#
# 注意: Qwen は GQA。Q は14個だが K/V は2個（7ヘッドで1組を共有）。

import gc

import einops
import pandas as pd
import torch

N_Q_QKV = 4     # 何問ぶん計算するか。OOM が出たら 2 に下げる

free_except("model", "tokens", "clean_tokens", "corrupted_tokens",
            "answer_tokens", "answer_token_indices", "prompts", "answers",
            "items", "items_all", "TASK_SETS", "TASK_SETS_OK", "NAMES", "DIMS",
            "head_labels", "ALL_HEAD_LABELS", "imshow", "line", "scatter",
            "FAMILY", "PREFIX", "device", "model_path")

ct = clean_tokens[:N_Q_QKV]
pt = corrupted_tokens[:N_Q_QKV]
ai = answer_token_indices[:N_Q_QKV]

QKV_NAMES = ["q", "k", "v", "z"]
_want = tuple(f"hook_{n}" for n in QKV_NAMES)
qkv_filter = lambda name: name.endswith(_want)


def _logit_diff(logits):
    if logits.ndim == 3:
        logits = logits[:, -1, :]
    return (logits.gather(1, ai[:, 0:1]) - logits.gather(1, ai[:, 1:2])).mean()


torch.set_grad_enabled(False)
with torch.no_grad():
    CLEAN_B = _logit_diff(model(ct)).item()
    CORR_B = _logit_diff(model(pt)).item()
print(f"clean 基準 {CLEAN_B:+.3f} / corrupted 基準 {CORR_B:+.3f}")


def metric(logits):
    return (_logit_diff(logits) - CORR_B) / (CLEAN_B - CORR_B)


def stack_from(cache):
    """{名前: [layer, batch, pos, head, d_head]} を CPU で返す"""
    out = {}
    for nm in QKV_NAMES:
        try:
            out[nm] = torch.stack(
                [cache[nm, l] for l in range(model.cfg.n_layers)], dim=0
            ).detach().float().cpu()
        except Exception as e:
            print(f"  {nm} は取得できず ({type(e).__name__})")
    return out


# ---- clean 側: 勾配なしで活性だけ取る ----
model.reset_hooks()
with torch.no_grad():
    _, c_cache = model.run_with_cache(ct, names_filter=qkv_filter)
CLEAN_ACT = stack_from(c_cache)
del c_cache
gc.collect(); torch.cuda.empty_cache()
print(f"clean 活性 取得  (GPU {torch.cuda.memory_allocated()/1e9:.2f} GB)")

# ---- corrupted 側: 活性と勾配を取る（逆伝播はここだけ）----
torch.set_grad_enabled(True)
model.reset_hooks()
grad_store = {}


def bwd_hook(act, hook):
    grad_store[hook.name] = act.detach()


model.add_hook(qkv_filter, bwd_hook, "bwd")
out, p_cache = model.run_with_cache(pt, names_filter=qkv_filter)
CORR_ACT = stack_from(p_cache)
val = metric(out)
val.backward()
model.reset_hooks()
torch.set_grad_enabled(False)

CORR_GRAD = {}
for nm in QKV_NAMES:
    keys = [k for k in grad_store if k.endswith(f"hook_{nm}")]
    keys.sort(key=lambda s: int(s.split(".")[1]))
    if len(keys) == model.cfg.n_layers:
        CORR_GRAD[nm] = torch.stack([grad_store[k] for k in keys], dim=0).float().cpu()

del out, p_cache, grad_store, val
gc.collect(); torch.cuda.empty_cache()
print(f"corrupted 活性と勾配 取得  (GPU {torch.cuda.memory_allocated()/1e9:.2f} GB)")
print()

# ---- [公式の式] 勾配 × (clean - corrupted)。1問ごとに残す ----
QKV = {}
for nm in QKV_NAMES:
    if nm not in CLEAN_ACT or nm not in CORR_ACT or nm not in CORR_GRAD:
        print(f"{nm}: 計算できず")
        continue
    a = einops.reduce(
        CORR_GRAD[nm] * (CLEAN_ACT[nm] - CORR_ACT[nm]),
        "layer batch pos head d_head -> layer head batch", "sum")
    QKV[nm] = a
    print(f"{nm}: {tuple(a.shape)}  (層, ヘッド, 問題)")

del CLEAN_ACT, CORR_ACT, CORR_GRAD
gc.collect(); torch.cuda.empty_cache()
print()


# ---- 写真の形で表示 ----
def show_qkv(qi, layers=None):
    layers = layers if layers is not None else range(model.cfg.n_layers)
    print("=" * 78)
    print(f"問題 #{qi}   Q/K/V の影響度")
    print(f"  {items[qi]['clean'][-70:]}")
    print("=" * 78)
    for L in layers:
        print(f"[層 {L:2d}]")
        for nm, jp in [("q", "Q"), ("k", "K"), ("v", "V"), ("z", "Z")]:
            if nm not in QKV:
                continue
            row = QKV[nm][L, :, qi]
            print(f"    {jp:<3}: " + "  ".join(f"{jp}{i}:{row[i]:+.3f}" for i in range(len(row))))
        print()


show_qkv(0, layers=range(model.cfg.n_layers))

# ---- CSV ----
rows = []
for nm, t in QKV.items():
    for L in range(t.shape[0]):
        for h in range(t.shape[1]):
            r = {"label": f"L{L}{nm.upper()}{h}", "layer": L, "kind": nm, "index": h}
            for qi in range(t.shape[2]):
                r[f"q{qi}"] = float(t[L, h, qi])
            r["mean"] = float(t[L, h].mean())
            r["consistency"] = float((t[L, h] > 0).float().mean())
            rows.append(r)

df_qkv = pd.DataFrame(rows)
csv_qkv = f"/content/qkv_{FAMILY}.csv"
df_qkv.to_csv(csv_qkv, index=False)
print(f"CSV 出力: {csv_qkv}  ({len(df_qkv)} 行)")
print()

top = df_qkv.reindex(df_qkv["mean"].abs().sort_values(ascending=False).index).head(20)
print("=" * 78)
print("Q/K/V で影響の大きい成分 上位20")
print("=" * 78)
print(f"  {'成分':<12}{'種別':<6}{'層':>3}{'平均':>9}{'一貫性':>8}")
for _, r in top.iterrows():
    print(f"  {r['label']:<12}{r['kind']:<6}{int(r['layer']):>3}{r['mean']:>9.3f}{r['consistency']:>8.0%}")

print()
print("V が大きい = 情報を運んでいる（運び屋）")
print("Q が大きい = どこを見るかを決めている（案内役）")
print("K が大きい = 見つけられる側の情報")


解放: 12 個 / GPU 使用中: 3.27 GB
clean 基準 +7.086 / corrupted 基準 -8.459
clean 活性 取得  (GPU 2.85 GB)
corrupted 活性と勾配 取得  (GPU 4.13 GB)

q: (24, 14, 4)  (層, ヘッド, 問題)
k: (24, 2, 4)  (層, ヘッド, 問題)
v: (24, 2, 4)  (層, ヘッド, 問題)
z: (24, 14, 4)  (層, ヘッド, 問題)

問題 #0   Q/K/V の影響度
  is stronger than Mark. Mark is stronger than Ann. The strongest one is
[層  0]
    Q  : Q0:+0.000  Q1:+0.000  Q2:+0.000  Q3:+0.000  Q4:+0.000  Q5:+0.000  Q6:+0.000  Q7:-0.000  Q8:+0.000  Q9:+0.000  Q10:+0.000  Q11:+0.000  Q12:+0.000  Q13:-0.000
    K  : K0:-0.000  K1:-0.000
    V  : V0:-0.004  V1:-0.000
    Z  : Z0:-0.000  Z1:-0.004  Z2:-0.000  Z3:+0.000  Z4:+0.000  Z5:+0.000  Z6:+0.000  Z7:+0.000  Z8:+0.000  Z9:-0.000  Z10:+0.000  Z11:+0.000  Z12:+0.000  Z13:-0.000

[層  1]
    Q  : Q0:+0.000  Q1:+0.000  Q2:+0.000  Q3:-0.000  Q4:+0.000  Q5:+0.000  Q6:+0.000  Q7:-0.000  Q8:-0.000  Q9:-0.000  Q10:-0.000  Q11:+0.000  Q12:-0.000  Q13:-0.000
    K  : K0:+0.000  K1:+0.000
    V  : V0:+0.000  V1:+0.000
    Z  : Z0:+0.000  Z1:-0.000  Z

In [8]:
import os, glob
print("現在地:", os.getcwd())
for p in glob.glob("/content/*.csv"):
    print(f"{p}  {os.path.getsize(p)/1e6:.1f} MB")


現在地: /content
/content/qkv_rel2.csv  0.1 MB
/content/influence_rel2.csv  26.9 MB


In [ ]:
# ========== STEP 15 : 1問 = 1ファイル（全成分） ==========
# 出力: /content/out/q001_rel0_000.csv のような CSV を1問につき1枚
#
# 1枚に入るもの（1問ぶんの全成分）
#   head_out  24層 × 14ヘッド =    336   ヘッド出力の寄与（順伝播）
#   q         24層 × 14       =    336   勾配ベース
#   k         24層 ×  2       =     48   GQA なので2個
#   v         24層 ×  2       =     48   GQA なので2個
#   z         24層 × 14       =    336   勾配ベース（ヘッド出力前）
#   neuron    24層 × 4864     = 116,736  FFN ニューロンの寄与（順伝播）
#   ─────────────────────────────────────
#   合計                        117,840 行 + embed/bias
#
# 値の意味: その成分が「正解ロジット − 不正解ロジット」にいくら足したか。
#           正なら正解を後押し、負なら逆方向。

import gc
import os
import re as _re

import einops
import pandas as pd
import torch
from transformer_lens import ActivationCache

OUT_DIR = "/content/out"
os.makedirs(OUT_DIR, exist_ok=True)

free_except("model", "TASK_SETS", "TASK_SETS_OK", "NAMES", "DIMS",
            "head_labels", "ALL_HEAD_LABELS", "imshow", "line", "scatter",
            "FAMILY", "PREFIX", "device", "model_path")

# ---- 全条件を通し番号でつなぐ（rel0 の1問目が qid=1）----
ORDER = ["rel0", "rel1", "rel2", "rel3", "rel4"]

# STEP4 を実行していれば TASK_SETS_OK を、無ければ TASK_SETS をその場で検査して使う
try:
    SRC = TASK_SETS_OK
    print("検査済みの問題集(TASK_SETS_OK)を使用")
except NameError:
    SRC = {}
    for cond, its in TASK_SETS.items():
        keep = []
        for t in its:
            ctk = model.to_tokens(t["clean"])
            ptk = model.to_tokens(t["corrupted"])
            c0 = model.to_tokens(t["correct"], prepend_bos=False)[0, 0].item()
            w0 = model.to_tokens(t["wrong"], prepend_bos=False)[0, 0].item()
            if ctk.shape[1] == ptk.shape[1] and c0 != w0:
                keep.append(t)
        if keep:
            SRC[cond] = keep
    print("TASK_SETS をその場で検査して使用")

QUESTIONS = []
for cond in ORDER:
    for j, t in enumerate(SRC.get(cond, [])):
        QUESTIONS.append(dict(qid=len(QUESTIONS) + 1, condition=cond, local_id=j, **t))
print(f"問題総数 {len(QUESTIONS)} 問")
for cond in ORDER:
    n = sum(1 for q in QUESTIONS if q["condition"] == cond)
    if n:
        ids = [q["qid"] for q in QUESTIONS if q["condition"] == cond]
        print(f"  {cond}: {n} 問  (qid {ids[0]}〜{ids[-1]})")
print()


def process_question(qid, verbose=True):
    """1問を処理して CSV を1枚書く"""
    q = QUESTIONS[qid - 1]

    ct = model.to_tokens(q["clean"])
    pt = model.to_tokens(q["corrupted"])
    c_id = model.to_tokens(q["correct"], prepend_bos=False)[0, 0].item()
    w_id = model.to_tokens(q["wrong"], prepend_bos=False)[0, 0].item()
    ai = torch.tensor([[c_id, w_id]], device=device)

    def ldiff(logits):
        if logits.ndim == 3:
            logits = logits[:, -1, :]
        return (logits.gather(1, ai[:, 0:1]) - logits.gather(1, ai[:, 1:2])).mean()

    # ---- 基本情報 ----
    torch.set_grad_enabled(False)
    with torch.no_grad():
        lg_c = model(ct)
        pred_id = lg_c[0, -1].argmax().item()
        LD_CLEAN = ldiff(lg_c).item()
        del lg_c
        LD_CORR = ldiff(model(pt)).item()
    is_correct = (pred_id == c_id)

    rows = []

    # ================= 順伝播1回: ヘッド出力 と ニューロン =================
    with torch.no_grad():
        _, cache = model.run_with_cache(ct)

    W_U = model.unembed.W_U
    dir_vec = (W_U[:, c_id] - W_U[:, w_id]).float()          # [d_model]

    stack, labels = cache.get_full_resid_decomposition(
        layer=-1, expand_neurons=True, apply_ln=True, pos_slice=-1,
        return_labels=True, project_output_onto=dir_vec)
    vals = stack.squeeze().float().cpu()                      # [component]
    del stack
    del cache
    gc.collect(); torch.cuda.empty_cache()

    for lab, v in zip(labels, vals.tolist()):
        m = _re.fullmatch(r"L(\d+)H(\d+)", lab)
        if m:
            rows.append(("head_out", int(m.group(1)), int(m.group(2)), lab, v)); continue
        m = _re.fullmatch(r"L(\d+)N(\d+)", lab)
        if m:
            rows.append(("neuron", int(m.group(1)), int(m.group(2)), lab, v)); continue
        m = _re.fullmatch(r"(\d+)_mlp_out", lab)
        if m:
            rows.append(("mlp_layer", int(m.group(1)), -1, lab, v)); continue
        rows.append(("other", -1, -1, lab, v))

    n_fwd = len(rows)

    # ================= 勾配: Q / K / V / Z =================
    QKV_NAMES = ["q", "k", "v", "z"]
    _want = tuple(f"hook_{n}" for n in QKV_NAMES)
    qkv_filter = lambda name: name.endswith(_want)

    def metric(logits):
        return (ldiff(logits) - LD_CORR) / (LD_CLEAN - LD_CORR)

    def stack_from(cache, tag):
        out = {}
        for nm in QKV_NAMES:
            try:
                out[nm] = torch.stack(
                    [cache[nm, l] for l in range(model.cfg.n_layers)], dim=0
                ).detach().float().cpu()
            except Exception as e:
                print(f"  [失敗] {tag} の '{nm}' を取り出せません: {type(e).__name__}: {e}")
                print(f"         キャッシュ内の名前(先頭10件): {list(cache.cache_dict.keys())[:10]}")
        return out

    model.reset_hooks()
    with torch.no_grad():
        _, c_cache = model.run_with_cache(ct, names_filter=qkv_filter)
    CLEAN_ACT = stack_from(c_cache, "clean")
    del c_cache
    gc.collect(); torch.cuda.empty_cache()

    torch.set_grad_enabled(True)
    model.reset_hooks()
    grad_store = {}

    def bwd_hook(act, hook):
        grad_store[hook.name] = act.detach()

    model.add_hook(qkv_filter, bwd_hook, "bwd")
    out, p_cache = model.run_with_cache(pt, names_filter=qkv_filter)
    CORR_ACT = stack_from(p_cache, "corrupted")
    metric(out).backward()
    model.reset_hooks()
    torch.set_grad_enabled(False)

    CORR_GRAD = {}
    if verbose:
        print(f"  勾配フック取得数: {len(grad_store)}")
    for nm in QKV_NAMES:
        keys = [k for k in grad_store if k.endswith(f"hook_{nm}")]
        keys.sort(key=lambda s: int(s.split(".")[1]))
        if len(keys) == model.cfg.n_layers:
            CORR_GRAD[nm] = torch.stack([grad_store[k] for k in keys], dim=0).float().cpu()
        else:
            print(f"  [失敗] '{nm}' の勾配が {len(keys)} 層ぶんしかありません "
                  f"(必要 {model.cfg.n_layers})")
            if keys:
                print(f"         例: {keys[:3]}")

    del out, p_cache, grad_store
    gc.collect(); torch.cuda.empty_cache()

    for nm in QKV_NAMES:
        missing = [t for t, d in [("clean活性", CLEAN_ACT), ("corrupted活性", CORR_ACT),
                                  ("勾配", CORR_GRAD)] if nm not in d]
        if missing:
            print(f"  [除外] '{nm}' は CSV に入りません。欠けているもの: {', '.join(missing)}")
            continue
        a = einops.reduce(CORR_GRAD[nm] * (CLEAN_ACT[nm] - CORR_ACT[nm]),
                          "layer batch pos head d_head -> layer head", "sum")
        for L in range(a.shape[0]):
            for h in range(a.shape[1]):
                rows.append((nm, L, h, f"L{L}{nm.upper()}{h}", float(a[L, h])))

    del CLEAN_ACT, CORR_ACT, CORR_GRAD
    gc.collect(); torch.cuda.empty_cache()

    # ================= 1枚のCSVにする =================
    df = pd.DataFrame(rows, columns=["kind", "layer", "index", "label", "value"])
    df.insert(0, "qid", q["qid"])
    df.insert(1, "condition", q["condition"])
    df["is_correct"] = int(is_correct)
    df["logit_diff_clean"] = round(LD_CLEAN, 4)
    df["logit_diff_corrupted"] = round(LD_CORR, 4)

    got = set(df["kind"].unique())
    want = {"head_out", "neuron", "q", "k", "v", "z"}
    lack = want - got
    if lack:
        print()
        print("!" * 74)
        print(f"警告: 次の種別が CSV に入っていません -> {sorted(lack)}")
        print("      上の [失敗] / [除外] の行に原因が出ています。")
        print("!" * 74)
        print()

    path = f"{OUT_DIR}/q{q['qid']:03d}_{q['condition']}_{q['local_id']:03d}.csv"
    df.to_csv(path, index=False)

    if verbose:
        print("=" * 74)
        print(f"qid {q['qid']}  ({q['condition']} の {q['local_id']} 番目)")
        print(f"  問題: ...{q['clean'][-70:]}")
        print(f"  正解 {q['correct']!r} / 不正解 {q['wrong']!r}")
        print(f"  予測 {model.to_string([pred_id])!r}  -> {'○ 正解' if is_correct else '× 不正解'}")
        print(f"  ロジット差  clean {LD_CLEAN:+.3f} / corrupted {LD_CORR:+.3f}")
        print("-" * 74)
        print(f"  行数 {len(df):,}  (順伝播 {n_fwd:,} + 勾配 {len(df) - n_fwd:,})")
        print("  内訳:")
        for k, n in df["kind"].value_counts().items():
            print(f"    {k:<10}{n:>8,}")
        print(f"  出力: {path}  ({os.path.getsize(path)/1e6:.1f} MB)")
        print("=" * 74)
        print()
        print("  影響の大きい成分 上位15:")
        top = df.reindex(df["value"].abs().sort_values(ascending=False).index).head(15)
        for _, r in top.iterrows():
            print(f"    {r['label']:<12}{r['kind']:<10}{r['value']:+.4f}")
    return path


# ---- 実行 ----
# RUN_ALL = False -> 1問だけ（動作確認用）
# RUN_ALL = True  -> 全問。既にファイルがある問題は飛ばすので、途中で落ちても再開できる
RUN_ALL = True

if not RUN_ALL:
    process_question(1)
else:
    import time
    t0 = time.time()
    done = skipped = failed = 0
    for q in QUESTIONS:
        path = f"{OUT_DIR}/q{q['qid']:03d}_{q['condition']}_{q['local_id']:03d}.csv"
        if os.path.exists(path):
            skipped += 1
            continue
        try:
            process_question(q['qid'], verbose=False)
            done += 1
        except Exception as e:
            failed += 1
            print(f"  qid {q['qid']} で失敗: {type(e).__name__}: {e}")
        if (done + skipped) % 10 == 0:
            el = time.time() - t0
            rate = el / max(done, 1)
            rest = (len(QUESTIONS) - done - skipped) * rate
            print(f"  {done + skipped}/{len(QUESTIONS)} 完了  "
                  f"経過 {el/60:.1f}分  残り約 {rest/60:.1f}分")

    print()
    print(f"新規 {done} / 既存を飛ばした {skipped} / 失敗 {failed}")
    print(f"所要 {(time.time()-t0)/60:.1f} 分")

    files = sorted(os.listdir(OUT_DIR))
    total = sum(os.path.getsize(os.path.join(OUT_DIR, f)) for f in files)
    print(f"出力 {len(files)} ファイル / 合計 {total/1e9:.2f} GB  ({OUT_DIR})")
